<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [4]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "662"
iso_country = "Saint Lucia"
year = 2020

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    "TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Saint Lucia': 

Number of trade records present in dataset: 15911
Number of partner countries present in dataset: 124
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$437.6m
Total weight within the dataset: 447.4 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,662,Saint Lucia,028,Antigua and Barbuda,Import,Sea,40299,040299 | Dairy produce; milk,0402,0402 | Milk and cream; conc,04,04 | Dairy produce; birds,2292.0,"Milk and cream, concentrated or containing add...",Agriculture,22.0,MIL - Dairy products,7.268859,0.0,7.268859,kg,1.118286,1.118286,2.567361,392.0,243.577432,0.625351,['Container'],['Container'],3 - Reporter and HS4 code,5.38,19,0.37,7.27,6.90,1. Primary Int. Inv. with Vess Type,0.003269,0.003263,0.003263,0.003263,0.003266,0.003266,0.003419,0.006060,0.003771,0.003501,0.003511,0.003599,0.003348,0.005935,0.003970,0.003636,0.003553,0.003714,0.003285,0.006453,0.004923,0.004305,0.003983,0.003871,0.003224,0.006862,0.005646,0.005291,0.004264,0.004148,0.003164,0.006953,0.006643,0.006700,0.004623,0.004358,0.003103,0.00681,0.006526,0.007151,0.005289,0.004438,0.003045,0.006075,0.006332,0.006635,0.004974,0.004490,0.002986,0.005580,0.005673,0.007076,0.005846,0.004520,2292
1,1,662,Saint Lucia,028,Antigua and Barbuda,Import,Sea,81340,"081340 | Fruit, edible; fruit",0813,"0813 | Fruit, dried, other",08,"08 | Fruit and nuts, edib",135.0,"Other fruit, dried",Agriculture,4.0,"V_F - Vegetables, fruit, nuts",16.850781,0.0,16.850781,kg,6.807204,6.807204,5.737691,392.0,243.577432,1.397572,['Container'],['Container'],3 - Reporter and HS4 code,4.79,6,0.77,16.85,16.08,1. Primary Int. Inv. with Vess Type,0.007305,0.007292,0.007293,0.007293,0.007299,0.007299,0.007641,0.013543,0.008429,0.007825,0.007846,0.008043,0.007482,0.013263,0.008872,0.008126,0.007940,0.008299,0.007343,0.014423,0.011001,0.009621,0.008901,0.008650,0.007206,0.015335,0.012619,0.011825,0.009530,0.009271,0.007071,0.015540,0.014847,0.014974,0.010332,0.009739,0.006934,0.01522,0.014586,0.015981,0.011821,0.009917,0.006805,0.013577,0.014151,0.014828,0.011116,0.010035,0.006673,0.012471,0.012678,0.015814,0.013064,0.010101,135


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [6]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $122.2m (27.9%), ... 

                             cifvalue  vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                           
02 | Meat and edible meat  15376129.1   7204649.9     12.6      7.0
22 | Beverages, spirits a  14145349.0  20484437.0     11.6     19.8
04 | Dairy produce; birds  13317181.5   5179032.8     10.9      5.0
19 | Preparations of cere  13003456.3   4602279.4     10.6      4.4
21 | Miscellaneous edible  10122744.7   2900966.5      8.3      2.8
07 | Vegetables and certa   7943246.9  13195225.8      6.5     12.7
16 | Meat, fish, crustace   7698912.8   1673042.1      6.3      1.6
20 | Preparations of vege   7386632.4   4886115.9      6.0      4.7
11 | Products of the mill   5900703.8  13092065.7      4.8     12.6
17 | Sugars and sugar con   5327424.6  14936155.5      4.4     14.4
23 | Food industries, res   4478967.8   3499235.3      3.7      3.4
15 | Animal, ve

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [15]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $120.6m (27.6%), ... 

                    cifvalue  vol_kg_vtw  pct_val  pct_vol
eora_desc                                                 
Food & Beverages  64787842.0  48752962.6     14.8     10.9
Agriculture       52876587.1  48457086.8     12.1     10.8
Fishing            2934013.5    440438.5      0.7      0.1



### 2.1.3 GTAP (16x headings)

In [8]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $101.5m (23.2%), ... 

                                 cifvalue  vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
OFD - Food products n.e.c.     49381084.4  28943729.6     11.3      6.5
OMT - Meat products n.e.c.     16708079.6   7462898.9      3.8      1.7
MIL - Dairy products           13108985.2   4845924.0      3.0      1.1
V_F - Vegetables, fruit, nuts   8909150.3  13823558.8      2.0      3.1
VOL - Vegetable oils and fats   4393552.3   3294532.8      1.0      0.7
CMT - Bovine meat prods         2634619.5    649198.6      0.6      0.1
SGR - Sugar                     2209090.7  10068647.0      0.5      2.3
PDR - Paddy rice                1573152.8   2732566.4      0.4      0.6
PCR - Processed rice             987428.8   1715163.9      0.2      0.4
OCR - Crops n.e.c.               766945.5    425550.5      0.2      0.1
FSH - Fishing                    650606.2    130575.1      0.1      0.

## 2.2 Comparison of results

In [9]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 27.9% 
GTAP 23.2% 
EORA 27.6%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [10]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [11]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_21391/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,662,Saint Lucia,028,Antigua and Barbuda,Import,Sea,40299,040299 | Dairy produce; milk,0402,0402 | Milk and cream; conc,04,04 | Dairy produce; birds,2292.0,"Milk and cream, concentrated or containing add...",Agriculture,22.0,MIL - Dairy products,7.268859,0.0,7.268859,kg,1.118286,1.118286,2.567361,392.0,243.577432,0.625351,['Container'],['Container'],3 - Reporter and HS4 code,5.38,19,0.37,7.27,6.90,1. Primary Int. Inv. with Vess Type,0.003269,0.003263,0.003263,0.003263,0.003266,0.003266,0.003419,0.006060,0.003771,0.003501,0.003511,0.003599,0.003348,0.005935,0.003970,0.003636,0.003553,0.003714,0.003285,0.006453,0.004923,0.004305,0.003983,0.003871,0.003224,0.006862,0.005646,0.005291,0.004264,0.004148,0.003164,0.006953,0.006643,0.006700,0.004623,0.004358,0.003103,0.00681,0.006526,0.007151,0.005289,0.004438,0.003045,0.006075,0.006332,0.006635,0.004974,0.004490,0.002986,0.005580,0.005673,0.007076,0.005846,0.004520,2292,Dairy
1,1,662,Saint Lucia,028,Antigua and Barbuda,Import,Sea,81340,"081340 | Fruit, edible; fruit",0813,"0813 | Fruit, dried, other",08,"08 | Fruit and nuts, edib",135.0,"Other fruit, dried",Agriculture,4.0,"V_F - Vegetables, fruit, nuts",16.850781,0.0,16.850781,kg,6.807204,6.807204,5.737691,392.0,243.577432,1.397572,['Container'],['Container'],3 - Reporter and HS4 code,4.79,6,0.77,16.85,16.08,1. Primary Int. Inv. with Vess Type,0.007305,0.007292,0.007293,0.007293,0.007299,0.007299,0.007641,0.013543,0.008429,0.007825,0.007846,0.008043,0.007482,0.013263,0.008872,0.008126,0.007940,0.008299,0.007343,0.014423,0.011001,0.009621,0.008901,0.008650,0.007206,0.015335,0.012619,0.011825,0.009530,0.009271,0.007071,0.015540,0.014847,0.014974,0.010332,0.009739,0.006934,0.01522,0.014586,0.015981,0.011821,0.009917,0.006805,0.013577,0.014151,0.014828,0.011116,0.010035,0.006673,0.012471,0.012678,0.015814,0.013064,0.010101,135,Fruits


## 3.3 Inspect the output

In [12]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $79.7m of $437.6m (18.2%)
Volume: 71.4 of 447.4 million kilograms (16.0%)
Value-to-Weight: 1.12 from 0.98 US$ per kg (+14.2%)
Average distance to market (volume-weighted): 1875.2 from 1790.3 nm (+4.7%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    356  30807705.6  24260903.6      7.0      5.4
1        Meat and Prep     46  13300940.9   6622115.6      3.0      1.5
2                Dairy    109  12975696.8   4988125.0      3.0      1.1
3           Vegetables    269  10781199.1  15521590.5      2.5      3.5
4      Sugar and Honey     61   5362738.9  14946280.1      1.2      3.3
5        Fats and Oils     86   4081263.9   2861225.3      0.9      0.6
6               Fruits    135   2282662.2   1716826.2      0.5      0.4
7  Unprocessed Cereals     20    147790.0    486504.1      0.0      0.1



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [17]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,8397412.6,14818334.2,9245302.8,8590352.8,8613303.7,8832946.3,7776483.9,17037965.1,16259689.9,16415670.8,11320809.7,10679021.9,7342883.1,13660702.2,13895088.1,17258974.5,14278218.8,11073482.1
1,Fats and Oils,2036161.7,3551801.9,2232905.7,2076886.2,2082474.6,2138137.7,1889472.4,4068635.8,3883047.7,3928337.9,2727093.9,2572789.5,1787068.6,3279644.2,3343666.9,4130522.8,3414331.5,2665496.1
2,Fruits,2123693.1,3762748.9,2342211.7,2174638.9,2180538.6,2235218.7,1965358.5,4318067.2,4125251.9,4160991.3,2870873.8,2706278.8,1854834.3,3465353.5,3522772.0,4393019.1,3629543.0,2806861.9
3,Meat and Prep,9405042.5,16668686.1,10374063.4,9631352.5,9657509.1,9899389.6,8703414.8,19126574.8,18273749.7,18430868.8,12716507.2,11986907.6,8213660.6,15350197.3,15603877.2,19463822.9,16079848.8,12432624.6
4,Processed Cereals,16471286.4,28882329.6,18090955.9,16828014.4,16872379.7,17309821.1,15270327.2,33291449.1,31745310.1,32089427.4,22143758.8,20875268.2,14431524.8,26691887.8,27151413.7,33576378.2,27817619.6,21636511.3
5,Sugar and Honey,9930304.6,16832545.3,10778756.6,10060053.1,10086714.6,10384581.1,9260071.9,19244290.4,18333674.1,18650315.6,13090380.9,12354509.8,8792137.3,15635080.4,16012715.6,19441937.2,16092348.9,12773316.4
6,Unprocessed Cereals,576803.1,997726.9,630152.6,587581.0,589076.0,604972.5,535986.6,1156649.3,1100411.6,1115452.1,770127.0,725791.9,507458.6,926480.7,943055.0,1153771.8,959291.9,751571.9
7,Vegetables,23031805.4,40708071.4,25372768.7,23569136.7,23632743.4,24231387.3,21323932.5,46768779.3,44654787.2,45069518.7,31084028.2,29311736.6,20130230.2,37515444.4,38149783.5,47453568.1,39228370.4,30396469.8


### 3.3.2 Total FEM costs by scenario (% cif)

In [16]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.49,0.07,0.01,0.02,0.03,0.71,0.65,0.67,0.27,0.22,0.49,0.50,0.76,0.53,0.29
1,Fats and Oils,0.37,0.05,0.01,0.01,0.02,0.53,0.49,0.50,0.21,0.17,0.37,0.38,0.57,0.40,0.22
2,Fruits,0.72,0.10,0.02,0.02,0.05,1.03,0.95,0.96,0.40,0.32,0.71,0.73,1.11,0.78,0.42
3,Meat and Prep,0.55,0.07,0.02,0.02,0.04,0.78,0.72,0.73,0.30,0.25,0.54,0.56,0.85,0.59,0.32
4,Processed Cereals,0.40,0.05,0.01,0.01,0.03,0.58,0.53,0.55,0.22,0.18,0.40,0.41,0.62,0.43,0.23
5,Sugar and Honey,1.29,0.16,0.02,0.03,0.08,1.86,1.69,1.75,0.71,0.58,1.28,1.35,1.99,1.36,0.74
6,Unprocessed Cereals,2.85,0.36,0.07,0.08,0.19,4.20,3.82,3.92,1.58,1.28,2.84,2.95,4.37,3.06,1.65
7,Vegetables,1.64,0.22,0.05,0.06,0.11,2.36,2.16,2.20,0.91,0.74,1.61,1.67,2.53,1.77,0.95
